# E-commerce

Generated from the FeatureMesh docs tutorial. This variant targets the FeatureMesh demos Jupyter environment.


## Set up FeatureMesh

Load the Jupyter magic and create a local `BatchClient` for the demos Jupyter environment. Run these cells once before the tutorial.


In [1]:
%load_ext featuremesh

In [2]:
from featuremesh import BatchClient, set_default

client = BatchClient()
set_default("client", client)
print("FeatureMesh BatchClient ready (local DuckDB)")

FeatureMesh: created local registry at /home/jovyan/featuremesh-demos/notebooks/demos/tutorials/standalone/analytics/featuremesh.db


FeatureMesh BatchClient ready (local DuckDB)


This tutorial builds a small retail feature model and uses it to answer questions that cross customers, orders, and line items.

The business questions are simple—How many orders did a customer place? Where was their latest order? What was it worth?—but each follows a different relationship shape. Learning those shapes here will make the rest of the analytics tutorials easier to read.

No previous domain tutorial is required. If `ENTITY()`, mappings, or `FOR` are completely new, the shorter [FeatureQL homepage companion](https://featuremesh.com/docs/tutorials/homepage/featureql) is a useful warm-up.

In the notebook, run the Data and Model sections before the relationship examples. If the notebook restarts, rerun those sections before continuing.


## Meet the retail data

We have three customers, four orders, and seven line items. John has two orders, while Jane and Jack have one each.

The setup also includes two small pre-aggregated helpers: one stores each customer's order IDs and a designated `last_order_id`, and another stores each order's total price. They keep the later array example focused. In production, you could derive or materialize the same values upstream.


In [3]:
%%featureql --client client --hide-dataframe

DROP FEATURES IF EXISTS IN FM.ECOMM UP TO LEVEL 9;

In [4]:
%%featureql --client client --hide-dataframe

CREATE SCHEMA IF NOT EXISTS tutorial_ecomm;
--
DROP TABLE IF EXISTS tutorial_ecomm.order_totals;
--
DROP TABLE IF EXISTS tutorial_ecomm.customer_orders;
--
DROP TABLE IF EXISTS tutorial_ecomm.items;
--
DROP TABLE IF EXISTS tutorial_ecomm.orders;
--
DROP TABLE IF EXISTS tutorial_ecomm.customers;
--
CREATE TABLE tutorial_ecomm.customers (
  customer_id BIGINT,
  name VARCHAR,
  time_create TIMESTAMP
);
--
INSERT INTO tutorial_ecomm.customers VALUES
  (100, 'John Doe', TIMESTAMP '2025-09-18 14:30:00'),
  (101, 'Jane Doe', TIMESTAMP '2024-12-08 09:45:00'),
  (102, 'Jack Doe', TIMESTAMP '2025-02-03 12:00:02');
--
CREATE TABLE tutorial_ecomm.orders (
  order_id BIGINT,
  order_customer_id BIGINT,
  order_city_name VARCHAR,
  time_create TIMESTAMP
);
--
INSERT INTO tutorial_ecomm.orders VALUES
  (200, 100, 'Barcelona', TIMESTAMP '2025-09-18 15:00:00'),
  (201, 101, 'Barcelona', TIMESTAMP '2024-12-08 09:45:00'),
  (202, 102, 'Barcelona', TIMESTAMP '2025-02-03 12:00:02'),
  (203, 100, 'Madrid', TIMESTAMP '2025-09-18 14:30:00');
--
CREATE TABLE tutorial_ecomm.items (
  item_id BIGINT,
  item_order_id BIGINT,
  item_product_name VARCHAR,
  price DECIMAL(10,2),
  quantity BIGINT
);
--
INSERT INTO tutorial_ecomm.items VALUES
  (300, 200, '400', 10.05, 2),
  (301, 200, '400', 11.05, 1),
  (302, 201, '401', 12.05, 3),
  (303, 202, '402', 13.05, 2),
  (304, 202, '402', 14.05, 1),
  (305, 202, '402', 15.05, 1),
  (306, 203, '400', 16.05, 1);
--
CREATE TABLE tutorial_ecomm.customer_orders (
  customer_id BIGINT,
  last_order_id BIGINT,
  orders BIGINT[]
);
--
INSERT INTO tutorial_ecomm.customer_orders VALUES
  (100, 200, [200, 203]),
  (101, 201, [201]),
  (102, 202, [202]);
--
CREATE TABLE tutorial_ecomm.order_totals (
  order_id BIGINT,
  price DECIMAL(10,2)
);
--
INSERT INTO tutorial_ecomm.order_totals VALUES
  (200, 31.15),
  (201, 36.15),
  (202, 55.20),
  (203, 16.05);

In [5]:
%%featureql --client client

SELECT customer_id, name
FROM tutorial_ecomm.customers
ORDER BY customer_id;

,customer_id,name
0,100,John Doe
1,101,Jane Doe
2,102,Jack Doe


After running the cell, inspect customer `100`: that is John, whose orders `200` and `203` give us a useful one-to-many case. The helper stores order `200` as John's `last_order_id`; this tutorial follows that stored field rather than deriving the latest order from timestamps.

## Give the tables business meaning

First declare customers, orders, and items as distinct entities. Their keys are all integers in storage, but the entity annotations stop a customer ID from being mistaken for an order ID.


In [6]:
%%featureql --client client

CREATE OR REPLACE FEATURES IN FM.ECOMM AS
PERSIST
    customers := ENTITY(),
    orders := ENTITY(),
    items := ENTITY(),
    customer_id := INPUT(BIGINT#customers),
    order_id := INPUT(BIGINT#orders),
    item_id := INPUT(BIGINT#items)
;

,feature_name,status,message
0,FM.ECOMM.CUSTOMERS,CREATED,Feature created as not exists
1,FM.ECOMM.ORDERS,CREATED,Feature created as not exists
2,FM.ECOMM.ITEMS,CREATED,Feature created as not exists
3,FM.ECOMM.CUSTOMER_ID,CREATED,Feature created as not exists
4,FM.ECOMM.ORDER_ID,CREATED,Feature created as not exists
5,FM.ECOMM.ITEM_ID,CREATED,Feature created as not exists


Now map the physical columns. `EXTERNAL_COLUMNS()` records the source table, column types, and key used to bind each row.


In [7]:
%%featureql --client client

CREATE OR REPLACE FEATURES IN FM.ECOMM AS
PERSIST
    tables.customers := EXTERNAL_COLUMNS(
        customer_id BIGINT#customers BIND TO customer_id,
        name VARCHAR,
        time_create TIMESTAMP
        FROM TABLE(tutorial_ecomm.customers)
    ),
    tables.orders := EXTERNAL_COLUMNS(
        order_id BIGINT#orders BIND TO order_id,
        order_customer_id BIGINT#customers,
        order_city_name VARCHAR,
        time_create TIMESTAMP
        FROM TABLE(tutorial_ecomm.orders)
    ),
    tables.items := EXTERNAL_COLUMNS(
        item_id BIGINT#items BIND TO item_id,
        item_order_id BIGINT#orders,
        item_product_name VARCHAR,
        price DECIMAL(10,2),
        quantity BIGINT
        FROM TABLE(tutorial_ecomm.items)
    ),
    tables.customer_orders := EXTERNAL_COLUMNS(
        customer_id BIGINT#customers BIND TO customer_id,
        last_order_id BIGINT#orders,
        orders ARRAY(BIGINT#orders)
        FROM TABLE(tutorial_ecomm.customer_orders)
    ),
    tables.order_totals := EXTERNAL_COLUMNS(
        order_id BIGINT#orders BIND TO order_id,
        price DECIMAL(10,2)
        FROM TABLE(tutorial_ecomm.order_totals)
    )
;

,feature_name,status,message
0,FM.ECOMM.TABLES.CUSTOMERS,CREATED,Feature created as not exists
1,FM.ECOMM.TABLES.ORDERS,CREATED,Feature created as not exists
2,FM.ECOMM.TABLES.ITEMS,CREATED,Feature created as not exists
3,FM.ECOMM.TABLES.CUSTOMER_ORDERS,CREATED,Feature created as not exists
4,FM.ECOMM.TABLES.ORDER_TOTALS,CREATED,Feature created as not exists


The persisted `TABLES.*` features represent raw sources. The shorter names—such as `customer_name`, `order_city`, and `item_price`—are the stable business features the queries will compose.

## See what a binding selects

Before crossing relationships, look at each population separately. Binding customer IDs produces one row per customer:


In [8]:
%%featureql --client client

RETURN
    customer_id,
    name := tables.customers[name],
    time_create := tables.customers[time_create]
FROM FM.ECOMM
FOR
    customer_id := BIND_COLUMNS(
        customer_id
        FROM TABLE(tutorial_ecomm.customers)
    )
ORDER BY customer_id
;

,FM.ECOMM.CUSTOMER_ID,NAME,TIME_CREATE
0,100,John Doe,2025-09-18 14:30:00
1,101,Jane Doe,2024-12-08 09:45:00
2,102,Jack Doe,2025-02-03 12:00:02


Binding order IDs changes the output grain to one row per order:


In [9]:
%%featureql --client client

RETURN
    order_id,
    order_customer_id := tables.orders[order_customer_id],
    city := tables.orders[order_city_name],
    time_create := tables.orders[time_create]
FROM FM.ECOMM
FOR
    order_id := BIND_COLUMNS(
        order_id
        FROM TABLE(tutorial_ecomm.orders)
    )
ORDER BY order_id
;

,FM.ECOMM.ORDER_ID,ORDER_CUSTOMER_ID,CITY,TIME_CREATE
0,200,100,Barcelona,2025-09-18 15:00:00
1,201,101,Barcelona,2024-12-08 09:45:00
2,202,102,Barcelona,2025-02-03 12:00:02
3,203,100,Madrid,2025-09-18 14:30:00


`FOR` therefore does more than supply values: it tells you which business objects the query is evaluating. Later, `NESTED` bindings will make child keys available inside a relationship without multiplying these outer rows.

## Align features on the same entity

Start at the customer grain and select two features already keyed by `CUSTOMER_ID`. Because they describe the same entity, FeatureQL aligns them directly; no `RELATED()` is needed.


In [10]:
%%featureql --client client

RETURN
    customer_id,
    customer_name := tables.customers[name],
    customer_orders := tables.customer_orders[orders]
FROM FM.ECOMM
FOR
    customer_id := BIND_VALUES(ARRAY(100, 101, 102))
ORDER BY customer_id
;

,FM.ECOMM.CUSTOMER_ID,CUSTOMER_NAME,CUSTOMER_ORDERS
0,100,John Doe,"[200, 203]"
1,101,Jane Doe,[201]
2,102,Jack Doe,[202]


John's row contains the order array **[200, 203]**. Jane and Jack each have a one-element array.

## Aggregate child rows back to a customer

Now count orders for each customer. The result starts at the customer key, groups the child rows by their customer foreign key, and returns the count to the matching customer.


In [11]:
%%featureql --client client

RETURN
    customer_id,
    num_orders := RELATED(
        SUM(1) GROUP BY (tables.orders[order_customer_id]) VIA customer_id
    )
FROM FM.ECOMM
FOR
    customer_id := BIND_VALUES(ARRAY(100, 101, 102)),
    NESTED order_id := BIND_VALUES(ARRAY(200, 201, 202, 203))
ORDER BY customer_id
;

,FM.ECOMM.CUSTOMER_ID,NUM_ORDERS
0,100,2
1,101,1
2,102,1


The result is John **2**, Jane **1**, and Jack **1**. In SQL, this shape usually requires a `GROUP BY` subquery followed by a join.

## Follow a foreign key to one parent row

Each customer row stores a `last_order_id`. Follow that key to the order entity and retrieve its city:


In [12]:
%%featureql --client client

DEFINE
    last_order_city := RELATED(
        tables.orders[order_city_name] VIA tables.customer_orders[last_order_id]
    )
RETURN
    customer_id,
    last_order_city
FROM FM.ECOMM
FOR
    customer_id := BIND_VALUES(ARRAY(100, 101, 102))
ORDER BY customer_id
;

,FM.ECOMM.CUSTOMER_ID,LAST_ORDER_CITY
0,100,Barcelona
1,101,Barcelona
2,102,Barcelona


All three designated last orders are in **Barcelona**. This is a lookup rather than an aggregation: one order ID identifies one order row.

## Continue across another relationship

The next question travels farther. Begin with each customer's stored `last_order_id`, find the items that belong to that order, then sum `price × quantity`.


In [13]:
%%featureql --client client

DEFINE
    last_order_id := tables.customer_orders[last_order_id],
    last_order_price := RELATED(
        SUM(tables.items[price] * tables.items[quantity]::DECIMAL)
        GROUP BY (tables.items[item_order_id]) VIA last_order_id
    )
RETURN
    customer_id,
    last_order_id,
    last_order_price
FROM FM.ECOMM
FOR
    customer_id := BIND_VALUES(ARRAY(100, 101, 102)),
    NESTED item_id := BIND_VALUES(ARRAY(300, 301, 302, 303, 304, 305, 306))
ORDER BY customer_id
;

,FM.ECOMM.CUSTOMER_ID,LAST_ORDER_ID,LAST_ORDER_PRICE
0,100,200,31.15000000
1,101,201,36.15000000
2,102,202,55.20000000


John's stored last order `200` totals **31.15**. Jane's totals **36.15**, and Jack's totals **55.20**. The output remains one row per customer even though the calculation reaches line items.

## Enrich arrays with `EXTEND()`

So far, `RELATED()` has returned one scalar per customer. To keep several related rows, start with the customer's order-ID array:

1. `ZIP()` turns the scalar IDs into rows.
2. `EXTEND()` follows each order ID to its total price.
3. The enriched array stays inside the customer row.


In [14]:
%%featureql --client client

RETURN
    customer_id,
    customer_orders := tables.customer_orders[orders],
    customer_orders_details := EXTEND(
        ZIP(customer_orders AS order_id)
        WITH tables.order_totals[price] AS order_price
    )
FROM FM.ECOMM
FOR
    customer_id := BIND_VALUES(ARRAY(100, 101, 102))
ORDER BY customer_id
;

,FM.ECOMM.CUSTOMER_ID,CUSTOMER_ORDERS,CUSTOMER_ORDERS_DETAILS
0,100,"[200, 203]","[{'order_id': 200, 'order_price': 31.15}, {'or..."
1,101,[201],"[{'order_id': 201, 'order_price': 36.15}]"
2,102,[202],"[{'order_id': 202, 'order_price': 55.20}]"


John now carries two enriched rows: order **200** at **31.15** and order **203** at **16.05**.

## Compose the relationship shapes

The final query combines customer name, enriched orders, stored last-order city, order count, and stored last-order value. Read each selected feature as a separate business question that FeatureQL aligns at the customer grain.


In [15]:
%%featureql --client client

DEFINE
    customer_name := tables.customers[name],
    customer_orders := tables.customer_orders[orders],
    last_order_id := tables.customer_orders[last_order_id],
    last_order_city := last_order_id.RELATED(tables.orders[order_city_name]),
    num_orders := ARRAY_COUNT(customer_orders),
    last_order_price := last_order_id.RELATED(
        SUM(tables.items[price] * tables.items[quantity]::DECIMAL)
        GROUP BY (tables.items[item_order_id])
    )
RETURN
    customer_id,
    customer_name,
    customer_orders,
    last_order_id,
    last_order_city,
    num_orders,
    last_order_price
FROM FM.ECOMM
FOR
    customer_id := BIND_VALUES(ARRAY(100, 101, 102)),
    NESTED item_id := BIND_VALUES(ARRAY(300, 301, 302, 303, 304, 305, 306))
ORDER BY customer_id
;

,FM.ECOMM.CUSTOMER_ID,CUSTOMER_NAME,CUSTOMER_ORDERS,LAST_ORDER_ID,LAST_ORDER_CITY,NUM_ORDERS,LAST_ORDER_PRICE
0,100,John Doe,"[200, 203]",200,Barcelona,2,31.15000000
1,101,Jane Doe,[201],201,Barcelona,1,36.15000000
2,102,Jack Doe,[202],202,Barcelona,1,55.20000000


Notice the bindings: customers define the outer rows; orders and items are `NESTED` because the relationship calculations need their keys internally.

## What's next

You have seen automatic same-entity alignment, child aggregation, parent lookup, a multi-hop calculation, and array enrichment.

Continue with [SaaS metrics](https://featuremesh.com/docs/tutorials/analytics/saas) to reuse those relationships in point-in-time measures and a persisted health score. Use the [`RELATED()`](https://featuremesh.com/docs/featureql/structural_operations/related) and [`EXTEND()`](https://featuremesh.com/docs/featureql/array_of_rows/extend) reference pages when you need the complete syntax.


---

Source tutorial: [/docs/tutorials/analytics/ecomm](https://featuremesh.com/docs/tutorials/analytics/ecomm)
